In [1]:
from bs4 import BeautifulSoup
from urllib import request
import pandas as pd
import datetime

from selenium import webdriver
import time

In [2]:
con2 = request.urlopen('https://finance.naver.com/item/main.naver?code=005930') #005930
con2.status

200

In [3]:
doc2 = BeautifulSoup(con2, 'html.parser')
# doc2

In [4]:
title_list = doc2.select('.wrap_company>h2>a') # >의 의미 : 바로 아래, 자식
# title_list = doc2.select('.wrap_company a') # 공백의 의미 : 저 안쪽 어딘가에, 자손
title_list

#select()의 결과는 무조건 list형태(ResultSet)
#결과가 하나의 태그라도 첫번째 인덱스로 꺼내주어야함.

[<a href="#" onclick="clickcr(this, 'sop.title', '', '', event);window.location.reload();">삼성전자</a>]

In [5]:
title_list[0]

<a href="#" onclick="clickcr(this, 'sop.title', '', '', event);window.location.reload();">삼성전자</a>

In [6]:
title_list[0].text.strip() #공백제거처리

'삼성전자'

In [7]:
title = title_list[0].get_text(strip=True)
title

'삼성전자'

In [8]:
# <span class="code">005930</span>
code_list = doc2.select('span.code')
code_list

[<span class="code">005930</span>]

In [9]:
code = code_list[0].text.strip()
code

'005930'

In [26]:
price_list = doc2.select('div.today')
#price_list #앗!! <span class="blind">95,100</span>가 숨겨있었음.

In [11]:
# 추출한 데이터에서 더 안으로 들어가면서 select로 계속 추출해도 됨.!!
# blind_price_list = price_list[0].select('span.blind')
# blind_price_list

In [12]:
price_list = doc2.select('div.today span.blind')
price_list

[<span class="blind">98,850</span>,
 <span class="blind">4,050</span>,
 <span class="blind">4.27</span>,
 <span class="blind">98,800</span>,
 <span class="blind">4,000</span>,
 <span class="blind">4.22</span>]

In [13]:
price_1 = price_list[0].text.strip()
price_1

'98,850'

In [14]:
# 숫자로 변환해야함.
price_today = int(price_1.replace(',' , ''))

type(price_today), price_today

(int, 98850)

In [15]:
# 전일가 price_yes, 고가 price_high

In [27]:
price_list2 = doc2.select('td.first')
# price_list2

In [17]:
price_list2 = doc2.select('td.first span.blind')
price_list2

[<span class="blind">94,800</span>,
 <span class="blind">97,800</span>,
 <span class="blind">94,800</span>,
 <span class="blind">96,700</span>]

In [18]:
price_2 = price_list2[0].text.strip()
price_2

'94,800'

In [19]:
price_yes = int(price_2.replace(",", ""))
price_yes

94800

In [22]:
price_list3 = doc2.select('td span.blind')
price_list3

[<span class="blind">94,800</span>,
 <span class="blind">99,000</span>,
 <span class="blind">123,200</span>,
 <span class="blind">8,212,954</span>,
 <span class="blind">97,800</span>,
 <span class="blind">97,150</span>,
 <span class="blind">805,737</span>,
 <span class="blind">94,800</span>,
 <span class="blind">99,000</span>,
 <span class="blind">123,200</span>,
 <span class="blind">6,738,870</span>,
 <span class="blind">96,700</span>,
 <span class="blind">95,700</span>,
 <span class="blind">656,922</span>]

In [23]:
price_3 = price_list3[1].text.strip() 
price_3

'99,000'

In [24]:
price_high = int(price_3.replace(",", ""))
price_high

99000

In [25]:
row = [code, title, price_today, price_yes, price_high]
row

['005930', '삼성전자', 98850, 94800, 99000]

In [ ]:
# com_list = pd.read_excel('상장법인목록.xls',
#                         header = 2,
#                         usecols= 'C',
#                         engine='xlrd'
#                         )
# com_list.head()
#===> xls가 아니라 xlsd(웹용 excel파일형식) 타입의 데이터였음.

In [ ]:

# 1) HTML 테이블로 읽어오기 (리스트 형태로 테이블들이 들어옴)
# tables = pd.read_html('상장법인목록.xls')  # 또는 .html 이라도 동일하게 동작

# len(tables)  # 몇 개의 테이블이 있는지 확인 (보통 1개)


In [ ]:
# df = tables[0]   # 첫 번째(유일한) 테이블

# com_list = df['종목코드']
# com_list

In [38]:
def get_finance_data(code):
    from bs4 import BeautifulSoup
    from urllib import request
    import pandas as pd
    import datetime

    con2 = request.urlopen('https://finance.naver.com/item/main.naver?code=' + code) #005930
    # print(con2.status)
    doc2 = BeautifulSoup(con2, 'html.parser')

    #회사명 추출
    title_list = doc2.select('.wrap_company>h2>a') # >의 의미 : 바로 아래, 자식
    # print(title_list)
    title = title_list[0].get_text(strip=True)
    print(title)

    # 코드 추출
    code_list = doc2.select('span.code')
    code = code_list[0].text.strip()
    print(code)

    # 시가 추출
    price_list = doc2.select('div.today span.blind')
    price_1 = price_list[0].text.strip()
    price_today = int(price_1.replace(',' , ''))
    print(price_today)

    # 어제가 추출
    price_list2 = doc2.select('td.first span.blind')
    price_2 = price_list2[0].text.strip()
    price_yes = int(price_2.replace(",", ""))
    print(price_yes)
    
    # 최고가 추출
    price_list3 = doc2.select('td span.blind')
    price_3 = price_list3[1].text.strip() 
    price_high = int(price_3.replace(",", ""))
    print(price_high)

    # 하나의 리스트로 묶자.
    row = [code, title, price_today, price_yes, price_high]
    print(row)  
    return row
    

In [33]:
get_finance_data("005930")

200
삼성전자
005930
98200
94800
99000
['005930', '삼성전자', 98200, 94800, 99000]


In [34]:
get_finance_data("035720")

200
카카오
035720
58500
58700
59500
['035720', '카카오', 58500, 58700, 59500]


In [35]:
code_list = ['005930', '035720', '035420', '042700']

In [36]:
for code in code_list:
    get_finance_data(code)
    print('----------------------------')

200
삼성전자
005930
98600
94800
99000
['005930', '삼성전자', 98600, 94800, 99000]
----------------------------
200
카카오
035720
58900
58700
59500
['035720', '카카오', 58900, 58700, 59500]
----------------------------
200
NAVER
035420
268000
262500
268500
['035420', 'NAVER', 268000, 262500, 268500]
----------------------------
200
한미반도체
042700
119900
120100
122500
['042700', '한미반도체', 119900, 120100, 122500]
----------------------------


In [39]:
rows = [] #여러 회사 정보 모을 리스트
for code in code_list:
    com = get_finance_data(code)
    rows.append(com)
    print('----------------------------')
rows

삼성전자
005930
98700
94800
99000
['005930', '삼성전자', 98700, 94800, 99000]
----------------------------
카카오
035720
58900
58700
59500
['035720', '카카오', 58900, 58700, 59500]
----------------------------
NAVER
035420
267000
262500
269000
['035420', 'NAVER', 267000, 262500, 269000]
----------------------------
한미반도체
042700
119900
120100
122500
['042700', '한미반도체', 119900, 120100, 122500]
----------------------------


[['005930', '삼성전자', 98700, 94800, 99000],
 ['035720', '카카오', 58900, 58700, 59500],
 ['035420', 'NAVER', 267000, 262500, 269000],
 ['042700', '한미반도체', 119900, 120100, 122500]]

In [40]:
df = pd.DataFrame(rows,  
                  columns=['code', 'name', 'price_today', 'price_yesterday', 'price_high']
                 )
df

,code,name,price_today,price_yesterday,price_high
0,005930,삼성전자,98700,94800,99000
1,035720,카카오,58900,58700,59500
2,035420,NAVER,267000,262500,269000
3,042700,한미반도체,119900,120100,122500


In [41]:
df.to_csv('naver_finance.csv',  index=False,  encoding='utf-8')

In [ ]:
#############################

In [ ]:
s1 = '95,100'
n1 = int(s1.replace(',', ''))
print(n1)      # 95100
print(type(n1))  # <class 'int'>


In [ ]:
s2 = '1,234,567'
n2 = int(s2.replace(',', ''))
print(n2)  # 1234567


In [ ]:
s3 = '95,100.5'
n3 = float(s3.replace(',', ''))
print(n3)  # 95100.5


In [ ]:
values = ['1,200', '3,450', '95,100']
nums = [int(v.replace(',', '')) for v in values]
print(nums)  # [1200, 3450, 95100]


In [ ]:
# df['price'] = df['price'].str.replace(',', '').astype(int)